# 06 - The nearest (OP) method explained

Computing a Green's function (GF) with the FK core is the expensive part of a
ShakerMaker run. The **nearest method** (a.k.a. the *OP pipeline*) exploits a
simple fact: in a horizontally layered crust, the GF between a source and a
receiver depends **only on the geometry**

$$ (d_h,\; z_\text{src},\; z_\text{rec}) $$

-- the horizontal distance, the source depth and the receiver depth -- not on
the absolute (x, y) position. So if two (source, receiver) pairs share the
same geometry (within tolerances `delta_h`, `delta_v_src`, `delta_v_rec`) they
share the **same** GF, and we only need to compute it **once**.

The pipeline has three stages:

- **Stage 0 -- `gen_pairs`**: pure geometry. Scan every (station, source) pair,
  group them into geometrically unique **slots**, and write a flat mapping
  `pair_to_slot[i_station * nsources + i_source] = k`.
- **Stage 1 -- `compute_gf`**: run the FK core **once per slot** `k`.
- **Stage 2 -- `run_fast`**: for every pair, fetch its slot's GF via the O(1)
  `pair_to_slot` index, convolve with the source time function and accumulate.

This notebook runs **only Stage 0** (geometry only -- fast, no FK core needed)
and visualizes which receivers get a **freshly computed** GF versus which ones
**reuse** an already-computed slot.

In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt

from shakermaker.shakermaker import ShakerMaker
from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.sl_extensions.SurfaceGrid import SurfaceGrid

## 1. Build a small surface grid and one source

A `SurfaceGrid` plane at z = 0 centred on (6, 8) km. With `nelems = [5, 5, 0]`
the grid has `(5+1) * (5+1) = 36` receivers plus one QA station at the centre.
A single Gaussian point source sits 2 km deep, so `nsources = 1` and the flat
pair index reduces to the station index.

In [ ]:
_m = 1e-3  # 1 m in km

crust = SCEC_LOH_1()
stf = Gaussian(t0=6 * 0.06, freq=1 / 0.06, M0=1e18 / 5e14 / 2, derivative=False)
source = PointSource([0, 0, 2], [0., 90., 0.], stf=stf)
fault = FaultSource([source], metadata={"name": "src"})

dx = 2.0
stations = SurfaceGrid([6.0, 8.0, 0.0], [5, 5, 0], [dx, dx, dx],
                       mode='plane', plane_z=0.0, metadata={"name": "grid"})

model = ShakerMaker(crust, fault, stations)
print("nstations:", stations.nstations)

# Receiver (x, y) coordinates in canonical station order.
coords = np.array([stations.get_station_by_id(i).x
                   for i in range(stations.nstations)])
src_xy = np.array(source.x)
coords[:3]

## Preview the model before running
Before executing, we inspect the crust (layered column + velocity profile), the source geometry, and its source time function (STF). Each figure is saved as a PNG in this folder.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.tools.plotting import SourcePlot

dt = 0.025                  # sampling step used when the GFs are later run

crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt          # sample the STF before plotting the source
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## 2. Run Stage 0 only (geometry)

`gen_pairs` writes a lightweight mapping file `<base>_map.h5`. It needs no FK
evaluation, so it is fast and safe to run anywhere. The tolerances control how
aggressively pairs are merged into shared slots.

In [ ]:
folder = "out_nearest_explained"
os.makedirs(folder, exist_ok=True)
gf_databasename = os.path.join(folder, "gf_database.h5")
base = gf_databasename.replace('.h5', '')

model.gen_pairs(gf_databasename,
                delta_h=2.5 * _m, delta_v_rec=2.5 * _m, delta_v_src=2.5 * _m,
                showProgress=True)

## 3. Open `<base>_map.h5` and read the mapping

The map file written by `gen_pairs` contains:

- `pair_to_slot` `(nstations * nsources,)` -- flat pair index -> slot index `k`
- `pairs_to_compute` `(n_slots, 2)` -- representative `[i_sta, i_src]` per slot
- `dh_of_pairs`, `dv_of_pairs`, `zrec_of_pairs`, `zsrc_of_pairs` -- slot geometry
- `delta_h`, `delta_v_rec`, `delta_v_src`, `nstations`, `nsources`

Because `nsources = 1`, `pair_to_slot[i]` is exactly the slot of station `i`.

In [ ]:
with h5py.File(base + "_map.h5", "r") as f:
    print("datasets:", list(f.keys()))
    pair_to_slot = f["pair_to_slot"][:]
    nsources = int(f["nsources"][()])
    nstations = int(f["nstations"][()])
    n_slots = f["pairs_to_compute"].shape[0]

print(f"nstations={nstations}  nsources={nsources}  n_slots={n_slots}")
print(f"reduction: {(1 - n_slots / len(pair_to_slot)) * 100:.1f}% "
      f"of GF evaluations skipped")
pair_to_slot

## 4. Classify each receiver: computed-fresh vs reused

Walking the pairs in canonical order, the **first** time a slot index appears
is the pair whose GF is actually computed by the FK core (Stage 1). Every
later pair pointing at that same slot **reuses** the stored GF. We mark each
receiver accordingly.

In [ ]:
is_fresh = np.zeros(len(pair_to_slot), dtype=bool)
seen = set()
for i, k in enumerate(pair_to_slot):
    if k not in seen:
        is_fresh[i] = True   # first occurrence of slot k -> GF computed here
        seen.add(k)
    # else: slot already computed -> this pair reuses it

# nsources == 1 so pair index == station index
fresh_recv = is_fresh
print(f"computed fresh: {fresh_recv.sum()}   reused: {(~fresh_recv).sum()}")

## 5. The key plot: calc vs reuse on the receiver surface

Each dot is a receiver, coloured by whether its GF is freshly computed
(orange) or reused from a previously computed slot (blue). The star is the
source's epicentre. Receivers at the same horizontal distance from the source
(same `d_h`) collapse onto a shared slot -- that is where the savings come
from.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

reuse = ~fresh_recv
ax.scatter(coords[reuse, 0], coords[reuse, 1], c="#3b6ea5", s=70,
           edgecolors="k", linewidths=0.4, label="reused GF")
ax.scatter(coords[fresh_recv, 0], coords[fresh_recv, 1], c="#e08214", s=90,
           edgecolors="k", linewidths=0.4, label="computed fresh")
ax.scatter([src_xy[0]], [src_xy[1]], marker="*", s=320, c="crimson",
           edgecolors="k", linewidths=0.6, label="source (epicentre)", zorder=5)

ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_title(f"Nearest method: {fresh_recv.sum()} computed / "
             f"{reuse.sum()} reused  (of {len(coords)} receivers)")
ax.set_aspect("equal")
ax.legend(loc="best")
ax.grid(True, ls=":", alpha=0.5)

plt.gcf().savefig("nearest_calc_vs_reuse.png", dpi=150, bbox_inches="tight")

## Takeaway

The orange receivers are the only ones the FK core ever runs for; the blue
receivers ride along for free. Tightening the tolerances
(`delta_h`, `delta_v_src`, `delta_v_rec`) creates more slots (more orange, more
accuracy, more cost); loosening them merges more pairs (more blue, cheaper,
more approximate). Stage 1 (`compute_gf`) then evaluates one GF per orange
slot, and Stage 2 (`run_fast`) reuses them across the whole grid.